# MRI Segmentation Project: Left Ventricle Detection

Notebook centré sur la visualisation des données du projet.

---

## Table of Contents

1. [Setup](#setup)
2. [Data Exploration](#data-exploration)
   - [Dataset Structure](#dataset-structure)
   - [Image Shape Distribution](#image-shape-distribution)
3. [Visualizing Patient Data](#visualizing-patient-data)
   - [Viewing Slices and Frames](#viewing-slices-and-frames)
4. [3D Visualization](#3d-visualization)

---

## Setup <a name="setup"></a>

In [ ]:
# Importing required libraries
import os
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = 'notebook'  # Use 'colab' if using Google Colab

# Enable inline plots for matplotlib
%matplotlib inline

---

## Data Exploration <a name="data-exploration"></a>

### Dataset Structure <a name="dataset-structure"></a>

The dataset is organized as follows:

III. AUTOMATIC LV S EGMENTATION
Our automatic LV segmentation algorithm consists of two
parts: 1) a soft endocardial contour extraction by iterative thresh-
olding, and 2) an epicardial contour extraction by active contour
modeling.
A. Endocardial Contour Extraction by Iterative Thresholding
A soft endocardial contour segmentation algorithm was used
[1], consisting of the following five steps.
1) Estimate the initial seed point.
2) Measure the mean and standard deviation of blood signal.
3) Compensate for coil sensitivity.
4) Estimate the myocardial (Myoc) signal intensity.
5) Segment the LV and measure the blood volume.
1) Step 1: Initial Seed Point Estimation: To identify a seed
point within the LV cavity, a circular Hough transform is per-
formed on the subtraction magnitude of images from phases
1 and 10 (roughly corresponding to diastole and systole, re-
spectively) of an initial mid-ventricular slice, selected by the
user [3], [12], [14], [23]. A seed propagation technique is ap-
plied for remaining slices and phases, where the seed point for
the current slice or phase image is determined by examining an
11 ×11 pixel window, whose center is the center of gravity of
the segmented LV region in the previous image. The energy of
each pixel in the window is calculated based on the distance
from the center of the window and the intensity difference from
the mean of LV region in the previous image
E(p) = 
√( 2σprev
w −1 |pCoG −p|
)2
+ ( Inew (p) −μprev)2 (1)
where p is the evaluated pixel position, Inew (p) is the intensity
of the evaluated pixel, pCoG is the center-of-gravity of LV region
in the previous image, w = 11 is the window width in number of
voxels, and σprev and μprev is the mean and standard deviation
of LV region in the previous image. The first term of the equation
penalizes pixels that are far away from the current seed point, and
the second term penalizes pixels that deviate from the current
blood intensity mean. The pixel with the lowest energy is chosen
as the seed point.
2) Step 2: Mean and Standard Deviation of Blood Signal
Estimation: Edge-based region-growing scheme is performed
from the initial seed point to find an exact LV region that is
nearly full-blood. The mean and standard deviation (LV m ean 
and LV std) of this exact LV region is calculated, and the voxels
visited in this step are considered as full-blood in all subsequent
calculations.
3) Step 3: Coil-Sensitivity Compensation: Images from MR
scanner have distortions depending on coil location that can
affect segmentation and volume calculations. To compensate
coil sensitivity across the area of the LV, the intensity variation
across the pool of full-blood voxels is modeled by a planar
surface, which is determined by fitting the plane to the voxels
of step 2 using least squares to attain the minimum residual.
4) Step 4: Myoc Signal Intensity Estimation: In order to esti-
mate the mean signal intensity, Myocm ean , of Myoc, successive
lower-bound threshold-based region-growing scheme is run for
an iteratively decreasing threshold, using the same seed as in
step 2. Eventually, region growth breaks through the Myoc,
“effusing” into surrounding structures (epicardial fat or fluids,
right ventricle, muscle, liver, etc.), causing a sudden increase
in the apparent segmented volume, which can be characterized
by a discontinuity in a ratio of consecutive volume growth.
The discontinuity of volume growth during decreasing region-
growing threshold is used to measure the threshold at which
region-growth has penetrated the Myoc. The threshold just be-
fore region-growth effuses into surrounding structures has been
previously shown [1] to occur at approximately 1 standard de-
viation away from the mean signal intensity, Myocm ean , of the 
Myoc, and is henceforth referred to as such.
5) Step 5: LV Segmentation and Volume Measurement: To
segment the LV region, region growing is performed again
for the last time within the image that slice or phase image
using an absolute threshold of Myocm ean  + n ×LV std, where
n = 2 . Accounting for Gaussian noise, the assignments ensure
LEE et al.: AUTOMATIC LEFT VENTRICLE SEGMENTATION USING ITHACA ON SHORT-AXIS CARDIAC MRI 907
Fig. 1. (a) Input image and segmented LV by iterative thresholding. (b) Input
image in polar coordinate. (c) Edge information by Canny detector. (d) LV in
polar coordinate. (e) Filtered edge information.
at least 99.4% of full-blood and full-myocardium voxels will
be weighted correctly. The total blood volume of the segmented
LV is determined by the summation of pixels as follows:
V = ∑
x
h(x)w(x) (2)
where V is volume and h(x) is the histogram of the LV. w(x)
is the weighting function associated with blood voxels of in-
tensity x and is used for calculating partial voxel effects. When
w(x) = 1 .0, the blood volume of the LV is calculated without
partial voxel effects. Fig. 1(a) depicts the segmented LV cavity
region by iterative thresholding. Since the absolute threshold
Myocm ean  + n ×LV std is larger than the intensity of the PTM
area, the presented algorithm excludes the PTM areas.
---

### Image Shape Distribution <a name="image-shape-distribution"></a>

In [ ]:
training_base_path = 'database/training/'
testing_base_path = 'database/testing/'

patient_folders = [folder for folder in os.listdir(training_base_path)]
print(f"Found {len(patient_folders)} patients in the training dataset.")

In [ ]:
image_shapes = []

for folder in patient_folders:
    image_path = os.path.join(training_base_path, folder, f'{folder}_4d.nii.gz')
    
    image = nib.load(image_path)
    data = image.get_fdata()
    
    image_shapes.append(data.shape)

# Convert the list to a NumPy array for easier manipulation
image_shapes = np.array(image_shapes)

In [ ]:
# Display the range of shapes
print("Image Shape Statistics:")
print(f"  Width: min = {image_shapes[:, 0].min()}, max = {image_shapes[:, 0].max()}")
print(f"  Height: min = {image_shapes[:, 1].min()}, max = {image_shapes[:, 1].max()}")
print(f"  Number of Slices: min = {image_shapes[:, 2].min()}, max = {image_shapes[:, 2].max()}")
print(f"  Time Frames: min = {image_shapes[:, 3].min()}, max = {image_shapes[:, 3].max()}")

In [ ]:
# 3D Scatter Plot of image shapes
fig = go.Figure(data=[
    go.Scatter3d(
        x=image_shapes[:, 0],
        y=image_shapes[:, 1],
        z=image_shapes[:, 2],
        mode='markers',
        marker=dict(size=5)
    )
])

fig.update_layout(
    title='Distribution of Image Shapes in Training Dataset',
    scene=dict(
        xaxis_title='Width',
        yaxis_title='Height',
        zaxis_title='Number of Slices'
    )
)

fig.show()

---

### Viewing Slices and Frames <a name="viewing-slices-and-frames"></a>

In [ ]:
# Specify the patient number to visualize
patient_number = 1
patient_id = f'patient{patient_number:03d}'

# Define paths to patient data
patient_path = os.path.join(training_base_path, patient_id)
image_4d_path = os.path.join(patient_path, f'{patient_id}_4d.nii.gz')
frame01_path = os.path.join(patient_path, f'{patient_id}_frame01.nii.gz')
frame01_gt_path = os.path.join(patient_path, f'{patient_id}_frame01_gt.nii.gz')

# Load images
image_4d = nib.load(image_4d_path).get_fdata()
frame01 = nib.load(frame01_path).get_fdata()
frame01_gt = nib.load(frame01_gt_path).get_fdata()

print(f"4D Image Shape: {image_4d.shape}")  # (Width, Height, Slices, Time Frames)
print(f"Frame01 Image Shape: {frame01.shape}")
print(f"Frame01 Ground Truth Shape: {frame01_gt.shape}")

In [ ]:
# Select a slice and time frame to visualize
slice_index = frame01.shape[2] // 2  # Middle slice

plt.imshow(frame01[:, :, slice_index], cmap='gray')
plt.title(f'Patient {patient_number} - Frame01 - Slice {slice_index}')
plt.axis('off')
plt.show()

In [ ]:
# Plot the ground truth segmentation for frame01
plt.imshow(frame01_gt[:, :, slice_index], cmap='gray')
plt.title(f'Patient {patient_number} - Frame01 Ground Truth - Slice {slice_index}')
plt.axis('off')
plt.show()

---

## 3D Visualization <a name="3d-visualization"></a>

Based on [Left ventricle using CNN](https://www.kaggle.com/code/abimbolasalawu/left-ventricle-using-cnn/notebook).


In [ ]:
time_frame = 0  # Adjust as needed

volume = image_4d[:, :, :, time_frame]
volume = volume.T  # Transpose to match Plotly's axis order
slices, height, width = volume.shape

# Create frames for animation
fig = go.Figure(frames=[
    go.Frame(
        data=go.Surface(
            z=k * np.ones((height, width)),
            surfacecolor=volume[slices - 1 - k],
            cmin=volume.min(),
            cmax=volume.max(),
            colorscale='Gray',
            showscale=False,
        ),
        name=str(k)
    )
    for k in range(slices)
])

# Add the initial surface
fig.add_trace(go.Surface(
    z=(slices - 1) * np.ones((height, width)),
    surfacecolor=volume[slices - 1],
    colorscale='Gray',
    cmin=volume.min(),
    cmax=volume.max(),
    colorbar=dict(thickness=20, ticklen=4)
))

# Define animation settings
def frame_args(duration):
    return {
        "frame": {"duration": duration},
        "mode": "immediate",
        "fromcurrent": True,
        "transition": {"duration": duration, "easing": "linear"},
    }

# Configure sliders and buttons
sliders = [{
    "pad": {"b": 10, "t": 60},
    "len": 0.9,
    "x": 0.1,
    "y": 0,
    "steps": [
        {
            "args": [[f.name], frame_args(0)],
            "label": str(k),
            "method": "animate",
        }
        for k, f in enumerate(fig.frames)
    ],
}]

fig.update_layout(
    title='3D Visualization of MRI Slices',
    width=800,
    height=800,
    scene=dict(
        zaxis=dict(range=[-1, slices], autorange=False),
        aspectratio=dict(x=1, y=1, z=1),
        xaxis_title='Width',
        yaxis_title='Height',
        zaxis_title='Slices',
    ),
    updatemenus=[
        {
            "buttons": [
                {
                    "args": [None, frame_args(50)],
                    "label": "&#9654;",  # Play symbol
                    "method": "animate",
                },
                {
                    "args": [[None], frame_args(0)],
                    "label": "&#9724;",  # Pause symbol
                    "method": "animate",
                },
            ],
            "direction": "left",
            "pad": {"r": 10, "t": 70},
            "type": "buttons",
            "x": 0.1,
            "y": 0,
        }
    ],
    sliders=sliders
)

fig.show()